# T20 Match Outcome Prediction

This notebook builds and compares multiple machine learning models for predicting T20 match outcomes.

## Models Implemented:
- Logistic Regression
- Random Forest
- Gradient Boosting
- XGBoost
- LightGBM
- CatBoost
- Neural Networks (TensorFlow)
- PyTorch Models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

sys.path.append(str(Path.cwd().parent / 'src'))

from modeling.match_predictor import MatchPredictor

print("Libraries imported successfully!")

## 1. Load Engineered Features

In [ ]:
# Load processed features
data_path = Path.cwd().parent / 'data' / 'processed' / 'features.csv'

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} samples with {df.shape[1]} features")
    display(df.head())
else:
    print("No features file found. Run notebook 02 first.")
    df = pd.DataFrame()

## 2. Prepare Data for Modeling

In [ ]:
if not df.empty:
    # Define target variable
    if 'result' in df.columns:
        y = (df['result'] == 'won').astype(int)
    else:
        print("Target variable 'result' not found")
        y = pd.Series()
    
    # Select features (exclude non-predictive columns)
    exclude_cols = ['match_id', 'date', 'result', 'team', 'opponent', 'venue']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    X = df[feature_cols].select_dtypes(include=[np.number])
    
    # Handle missing values
    X = X.fillna(X.mean())
    
    print(f"\nFeature matrix shape: {X.shape}")
    print(f"Target distribution:\n{y.value_counts()}")
    print(f"\nFeatures selected: {len(X.columns)}")
else:
    X = pd.DataFrame()
    y = pd.Series()

## 3. Train Traditional ML Models

In [ ]:
if not X.empty and not y.empty:
    # Initialize predictor with all models
    predictor = MatchPredictor(model_type='all')
    
    # Train models
    print("Training all models...\n")
    results = predictor.train(X, y, test_size=0.2)
    
    # Display results
    results_df = pd.DataFrame(results).T
    results_df = results_df.sort_values('accuracy', ascending=False)
    
    print("\n" + "="*70)
    print("MODEL PERFORMANCE COMPARISON")
    print("="*70)
    display(results_df)
else:
    print("Insufficient data for model training")

## 4. Visualize Model Performance

In [ ]:
if not X.empty and 'results_df' in locals():
    # Performance comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy comparison
    results_df['accuracy'].plot(kind='barh', ax=axes[0, 0], color='skyblue')
    axes[0, 0].set_title('Model Accuracy Comparison')
    axes[0, 0].set_xlabel('Accuracy')
    
    # F1 Score comparison
    results_df['f1_score'].plot(kind='barh', ax=axes[0, 1], color='lightgreen')
    axes[0, 1].set_title('F1 Score Comparison')
    axes[0, 1].set_xlabel('F1 Score')
    
    # Cross-validation scores
    if 'cv_mean' in results_df.columns:
        results_df['cv_mean'].plot(kind='barh', ax=axes[1, 0], color='coral')
        axes[1, 0].set_title('Cross-Validation Accuracy')
        axes[1, 0].set_xlabel('CV Mean Accuracy')
    
    # ROC AUC comparison
    if 'roc_auc' in results_df.columns:
        results_df['roc_auc'].plot(kind='barh', ax=axes[1, 1], color='mediumpurple')
        axes[1, 1].set_title('ROC AUC Comparison')
        axes[1, 1].set_xlabel('ROC AUC')
    
    plt.tight_layout()
    plt.show()

## 5. Feature Importance Analysis

In [ ]:
if not X.empty and 'predictor' in locals():
    # Get feature importance from tree-based models
    importance_dict = predictor.get_feature_importance(top_n=15)
    
    if importance_dict:
        n_models = len(importance_dict)
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()
        
        for idx, (model_name, importance_df) in enumerate(importance_dict.items()):
            if idx < len(axes):
                # Map feature indices to names
                importance_df['feature_name'] = importance_df['feature'].map(
                    lambda x: X.columns[x] if x < len(X.columns) else f'Feature {x}'
                )
                
                axes[idx].barh(importance_df['feature_name'], importance_df['importance'])
                axes[idx].set_title(f'{model_name} - Top Features')
                axes[idx].set_xlabel('Importance')
        
        # Hide unused subplots
        for idx in range(len(importance_dict), len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()

## 6. Deep Learning Model (TensorFlow)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

if not X.empty and not y.empty:
    # Prepare data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Build neural network
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC()]
    )
    
    print("\nTraining Neural Network...")
    history = model.fit(
        X_train_scaled, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    # Evaluate
    test_loss, test_acc, test_auc = model.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"\nNeural Network Performance:")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for neural network training")

## 7. PyTorch Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class CricketPredictor(nn.Module):
    def __init__(self, input_size):
        super(CricketPredictor, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x

if not X.empty and not y.empty:
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)
    
    # Create model
    torch_model = CricketPredictor(X_train_scaled.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=0.001)
    
    # Training
    print("\nTraining PyTorch Model...")
    epochs = 50
    batch_size = 32
    
    dataset = TensorDataset(X_train_tensor, y_train_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        torch_model.train()
        for batch_X, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = torch_model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    
    # Evaluation
    torch_model.eval()
    with torch.no_grad():
        predictions = torch_model(X_test_tensor)
        predictions = (predictions > 0.5).float()
        accuracy = (predictions == y_test_tensor).float().mean()
        print(f"\nPyTorch Model Test Accuracy: {accuracy.item():.4f}")
else:
    print("Insufficient data for PyTorch model training")

## 8. Model Comparison Summary

In [ ]:
# Create comprehensive comparison
if 'results_df' in locals():
    comparison = results_df[['accuracy', 'f1_score', 'cv_mean']].copy()
    
    # Add deep learning results if available
    if 'test_acc' in locals():
        comparison.loc['TensorFlow NN'] = [test_acc, np.nan, np.nan]
    if 'accuracy' in locals() and isinstance(accuracy, torch.Tensor):
        comparison.loc['PyTorch NN'] = [accuracy.item(), np.nan, np.nan]
    
    print("\n" + "="*70)
    print("FINAL MODEL COMPARISON")
    print("="*70)
    display(comparison.sort_values('accuracy', ascending=False))
    
    # Recommendation
    best_model = comparison['accuracy'].idxmax()
    best_accuracy = comparison['accuracy'].max()
    
    print(f"\n🏆 RECOMMENDED MODEL: {best_model}")
    print(f"   Accuracy: {best_accuracy:.4f}")

## 9. Save Best Model

In [ ]:
import joblib

if 'predictor' in locals() and predictor.best_model:
    # Save model
    model_path = Path.cwd().parent / 'video_analysis' / 'models'
    model_path.mkdir(exist_ok=True)
    
    # Save predictor object
    joblib.dump(predictor, model_path / 'best_predictor.pkl')
    print(f"Best model ({predictor.best_model}) saved to: {model_path / 'best_predictor.pkl'}")
    
    # Save scaler
    joblib.dump(scaler, model_path / 'scaler.pkl')
    print(f"Scaler saved to: {model_path / 'scaler.pkl'}")

## Next Steps

1. **Hyperparameter Tuning** - Optimize best performing models
2. **Ensemble Methods** - Combine multiple models
3. **Model Deployment** - Integrate into Streamlit dashboard
4. **Real-time Predictions** - Set up live match prediction pipeline